In [21]:
!pip install langchain-google-genai faiss-cpu tiktoken streamlit


In [22]:
from google.colab import files
uploaded = files.upload()


Saving National Flag.txt to National Flag (1).txt


In [23]:
pip install -U langchain-community

In [25]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = TextLoader("National Flag.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(documents)
print(f"Chunks created: {len(docs)}")


Chunks created: 3


In [36]:
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print("-" * 20)

--- Chunk 1 ---
The Indian National Flag is a symbol of pride, unity, and sovereignty. Called the 'Tiranga', it consists of three horizontal stripes: saffron on top, white in the middle, and green at the bottom.

Saffron stands for courage and sacrifice, white symbolizes peace and truth, while green represents faith and fertility. In the center is the navy-blue Ashoka Chakra with 24 spokes.
--------------------
--- Chunk 2 ---
The Chakra signifies the eternal wheel of law. It’s inspired by the Lion Capital of Ashoka, one of India’s historical emblems.

The flag was officially adopted on July 22, 1947, days before India gained independence from British rule.

The Flag Code of India governs its usage. It must always be treated with respect. Disrespecting it is a punishable offense under Indian law.
--------------------
--- Chunk 3 ---
National days like Independence Day and Republic Day witness widespread hoisting of the flag, evoking patriotism among the people.

The flag not only symbo

In [28]:
import os
os.environ["GOOGLE_API_KEY"] = "Your api key here"  # Replace with your actual key


In [29]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
db = FAISS.from_documents(docs, embeddings)
db.save_local("national_flag_index")


In [30]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant that answers questions using the provided context.
If you don't know the answer, say 'I don't know.'

Context:
{context}

Question:
{question}

Answer:"""
)

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", temperature=0.2)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=db.as_retriever(),
    memory=memory,
    combine_docs_chain_kwargs={"prompt": prompt_template}
)

In [31]:
query = "What is the significance of saffron in the flag?"
response = qa_chain.run(query)
print("Answer:", response)


Answer: Saffron in the Indian flag stands for courage and sacrifice.


In [32]:
query = "which country national flag we were talking about?"
response = qa_chain.run(query)
print("Answer:", response)

Answer: India's national flag contains saffron.


In [35]:
query = "which country is east neighbour of india?"
response = qa_chain.run(query)
print("Answer:", response)

Answer: I don't know.


In [33]:
import re
import evaluate

ground_truth = [
    {"question": "What does the saffron color in the Indian flag represent?", "answer": "Courage and sacrifice"},
    {"question": "When was the Indian National Flag adopted?", "answer": "July 22, 1947"},
    {"question": "What is in the center of the Indian flag?", "answer": "Ashoka Chakra"}
]

def normalize(text):
    return re.sub(r"[^\w\s]", "", text.lower().strip())

def compute_em(pred, truth):
    return int(normalize(pred) == normalize(truth))

def compute_f1(pred, truth):
    pred_tokens = normalize(pred).split()
    truth_tokens = normalize(truth).split()
    common = set(pred_tokens) & set(truth_tokens)
    if not common:
        return 0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)
    return 2 * (precision * recall) / (precision + recall)

# Get predictions
predictions = []
for item in ground_truth:
    pred = qa_chain.run(item["question"])
    predictions.append({
        "question": item["question"],
        "predicted": pred,
        "ground_truth": item["answer"],
        "EM": compute_em(pred, item["answer"]),
        "F1": compute_f1(pred, item["answer"])
    })


In [18]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.0 MB/s eta 0:00:00


In [34]:
# Print individual scores
for p in predictions:
    print(f"Q: {p['question']}")
    print(f"Predicted: {p['predicted']}")
    print(f"Ground Truth: {p['ground_truth']}")
    print(f"Exact Match: {p['EM']}, F1 Score: {p['F1']:.2f}\n")

# Averages
avg_em = sum(p["EM"] for p in predictions) / len(predictions)
avg_f1 = sum(p["F1"] for p in predictions) / len(predictions)

print(f"📈 Average Exact Match: {avg_em:.2f}")
print(f"📈 Average F1 Score: {avg_f1:.2f}")


Q: What does the saffron color in the Indian flag represent?
Predicted: Saffron in the Indian national flag stands for courage and sacrifice.
Ground Truth: Courage and sacrifice
Exact Match: 0, F1 Score: 0.43

Q: When was the Indian National Flag adopted?
Predicted: The Indian national flag was officially adopted on July 22, 1947.
Ground Truth: July 22, 1947
Exact Match: 0, F1 Score: 0.43

Q: What is in the center of the Indian flag?
Predicted: The navy-blue Ashoka Chakra with 24 spokes is in the center of the Indian national flag.
Ground Truth: Ashoka Chakra
Exact Match: 0, F1 Score: 0.22

📈 Average Exact Match: 0.00
📈 Average F1 Score: 0.36
